### From Silver To Gold: Aggregation and KPI Tables

In [0]:
import pyspark.sql.functions as F
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("catalog_name", "ecommerce", "Catalog Name")
dbutils.widgets.text("storage_account_name", "ecommcistorage05", "Storage Account Name")
dbutils.widgets.text("container_name", "ecomm-raw-data", "Container Name")

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
storage_account_name = dbutils.widgets.get("storage_account_name")
container_name = dbutils.widgets.get("container_name")

In [0]:
df_batch = spark.read.table(f"{catalog_name}.silver.slv_returns")
display(df_batch.limit(20))

return_ts,order_dt,order_id,reason,_rescued_data,ingest_timestamp,source_file,processed_time
2025-05-13T17:15:57.000Z,2025-04-30,540303,WRONG ITEM,null,2026-09-27T13:38:01.279Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_returns/landing/returns_2025-05.csv,2026-09-28T11:00:41.920Z
2025-05-21T01:48:49.000Z,2025-05-09,549906,OTHER,null,2026-09-27T13:38:01.279Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_returns/landing/returns_2025-05.csv,2026-09-28T11:00:41.920Z
2025-05-23T16:21:34.000Z,2025-05-11,552945,WRONG ITEM,null,2026-09-27T13:38:01.279Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_returns/landing/returns_2025-05.csv,2026-09-28T11:00:41.920Z
2025-05-26T20:52:06.000Z,2025-05-11,553991,WRONG ITEM,null,2026-09-27T13:38:01.279Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_returns/landing/returns_2025-05.csv,2026-09-28T11:00:41.920Z
2025-05-31T08:18:34.000Z,2025-05-20,563121,WRONG ITEM,null,2026-09-27T13:38:01.279Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_returns/landing/returns_2025-05.csv,2026-09-28T11:00:41.920Z
2025-01-04T13:01:42.000Z,2024-12-25,399527,LATE DELIVERY,null,2026-09-27T13:38:01.279Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_returns/landing/returns_2025-01.csv,2026-09-28T11:00:41.920Z
2025-01-04T20:10:00.000Z,2024-12-24,398122,DAMAGED,null,2026-09-27T13:38:01.279Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_returns/landing/returns_2025-01.csv,2026-09-28T11:00:41.920Z
2025-01-06T01:35:09.000Z,2024-12-28,401964,LATE DELIVERY,null,2026-09-27T13:38:01.279Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_returns/landing/returns_2025-01.csv,2026-09-28T11:00:41.920Z
2025-01-08T06:14:50.000Z,2025-01-01,406838,QUALITY ISSUE,null,2026-09-27T13:38:01.279Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_returns/landing/returns_2025-01.csv,2026-09-28T11:00:41.920Z
2025-01-14T02:18:22.000Z,2025-01-04,410921,DAMAGED,null,2026-09-27T13:38:01.279Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_returns/landing/returns_2025-01.csv,2026-09-28T11:00:41.920Z


In [0]:
spark.sql(f"USE CATALOG {catalog_name}")

DataFrame[]

In [0]:
# readChangeFeed flag is used to read the change feed (_change_type column mainly)
df = spark.readStream \
.format("delta") \
.option("readChangeFeed", "true") \
.table(f"{catalog_name}.silver.slv_returns")

In [0]:
df_union = df.filter("_change_type IN ('insert', 'update_postimage')")

### Add `date_id` column: yyyyMMdd format from order_dt

In [0]:
from pyspark.sql.types import IntegerType
df_union = df.withColumn("date_id",F.date_format(F.col("order_dt"),"yyyyMMdd").cast(IntegerType()))

### Calculate `return_days` = difference in days between return_ts and order_dt.

In [0]:
df_union = df_union.withColumn("return_days",F.datediff(F.col("return_ts"),F.col("order_dt")))

###Create policy compliance flags:
###within_policy → 1 if return_days <= 15, else 0.
###is_late_return → 1 if return_days > 15, else 0. 

In [0]:
df_union = df_union.withColumn("within_policy",F.when(F.col("return_days")<=15,1).otherwise(0))
df_union = df_union.withColumn("is_late_return",F.when(F.col("return_days")>15,1).otherwise(0))

In [0]:
df = df_union.select('date_id', 'return_ts', 'order_dt', 'order_id', 'reason','return_days', 'within_policy','is_late_return', '_rescued_data', 'ingest_timestamp', 'source_file')

### Gold checkpoint

In [0]:
gold_check_point = f'abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoint/gold/fact_returns/'

In [0]:
def upsert_to_gold(microBatchDF, batchId):
    table_name = f"{catalog_name}.gold.gld_fact_returns"
    if not spark.catalog.tableExists(table_name):
        print("creating new table")
        microBatchDF.write.format("delta").mode("overwrite").saveAsTable(table_name)
        spark.sql(
            f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
        )
    else:
        deltaTable = DeltaTable.forName(spark, table_name)
        deltaTable.alias("gold_table").merge(
            microBatchDF.alias("batch_table"),
            "gold_table.order_id = batch_table.order_id"
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()


In [0]:
df.writeStream.trigger(availableNow=True).foreachBatch(
    upsert_to_gold
).format("delta").option("checkpointLocation", gold_check_point).option(
    "mergeSchema", "true"
).outputMode(
    "update"
).trigger(
    once=True
).start().awaitTermination()